In [1]:
import time

notebook_start_time = time.time()

# Set up environment

In [ ]:
def install_dependencies() -> None:
    # !pip install --upgrade uv
    # !uv pip install --all-extras --system --requirement pyproject.toml
    pass


⛳️ Local environment
Adding the following directory to the PYTHONPATH: c:\Users\SUNIL KUMAR\Downloads\krish_naik\H_and_m


# 👩🏻‍🔬 Feature pipeline: Computing features

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import torch
from loguru import logger
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

from recsys import hopsworks_integration
from recsys.config import settings
from recsys.features.articles import (
    compute_features_articles,
    generate_embeddings_for_dataframe,
)
from recsys.features.customers import DatasetSampler, compute_features_customers
from recsys.features.interaction import generate_interaction_data
from recsys.features.ranking import compute_ranking_dataset
from recsys.features.transactions import compute_features_transactions
from recsys.hopsworks_integration import feature_store
from recsys.raw_data_sources import h_and_m as h_and_m_raw_data


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: c:\Users\SUNIL KUMAR\Downloads\krish_naik\personalized-recommender-course\.venv\Scripts\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\SUNIL KUMAR\Downloads\krish_naik\personalized-recommender-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Constants

These are the default settings used across the lessons. You can always override them in the `.env` file that sits at the root of the repository:

In [4]:
pprint(dict(settings))

{'CUSTOMER_DATA_SIZE': <CustomerDatasetSize.SMALL: 'SMALL'>,
 'CUSTOM_HOPSWORKS_INFERENCE_ENV': 'custom_env_name',
 'FEATURES_EMBEDDING_MODEL_ID': 'all-MiniLM-L6-v2',
 'HOPSHOST': 'eu-west.cloud.hopsworks.ai',
 'HOPSWORKS_API_KEY': SecretStr('**********'),
 'HOPSWORKS_PROJECT_NAME': 'hm_recommender',
 'OPENAI_API_KEY': SecretStr('**********'),
 'OPENAI_MODEL_ID': 'gpt-4o-mini',
 'RANKING_DATASET_VALIDATON_SPLIT_SIZE': 0.1,
 'RANKING_EARLY_STOPPING_ROUNDS': 5,
 'RANKING_ITERATIONS': 100,
 'RANKING_LEARNING_RATE': 0.2,
 'RANKING_MODEL_TYPE': 'ranking',
 'RANKING_SCALE_POS_WEIGHT': 10,
 'RECSYS_DIR': WindowsPath('c:/Users/SUNIL KUMAR/Downloads/krish_naik/H_and_m/recsys'),
 'TWO_TOWER_DATASET_TEST_SPLIT_SIZE': 0.1,
 'TWO_TOWER_DATASET_VALIDATON_SPLIT_SIZE': 0.1,
 'TWO_TOWER_LEARNING_RATE': 0.01,
 'TWO_TOWER_MODEL_BATCH_SIZE': 2048,
 'TWO_TOWER_MODEL_EMBEDDING_SIZE': 16,
 'TWO_TOWER_NUM_EPOCHS': 10,
 'TWO_TOWER_WEIGHT_DECAY': 0.001}


The most important one is the dataset size.

Choosing a different dataset size will impact the time it takes to run everything and the quality of the final models. We suggest using a small dataset size when running this the first time.

Suported user dataset sizes:

In [5]:
DatasetSampler.get_supported_sizes()

{<CustomerDatasetSize.LARGE: 'LARGE'>: 50000,
 <CustomerDatasetSize.MEDIUM: 'MEDIUM'>: 5000,
 <CustomerDatasetSize.SMALL: 'SMALL'>: 1000}

## <span style="color:#ff5f27">🔮 Connect to Hopsworks Feature Store </span>

In [5]:
project, fs = hopsworks_integration.get_feature_store()

2026-06-08 22:40:46.367 | INFO     | recsys.hopsworks_integration.feature_store:get_feature_store:37 - Loging to Hopsworks using HOPSWORKS_API_KEY env var.


2026-06-08 22:40:46,369 INFO: Initializing external client
2026-06-08 22:40:46,369 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


2026-06-08 22:40:53,578 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/33047


# The H&M dataset

To show how a recommender system using the two tower architecture works, we will use the [H&M Personalized Fashion Recommendations](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations) dataset.

It consists of:
- articles
- customers
- transactions

# 🗄️ Articles data

The **article_id** and **product_code** serve different purposes in the context of H&M's product database:

- **Article ID**: This is a unique identifier assigned to each individual article within the database. It is typically used for internal tracking and management purposes. Each distinct item or variant of a product (e.g., different sizes or colors) would have its own unique article_id.

- **Product Code**: This is also a unique identifier, but it is associated with a specific product or style rather than individual articles. It represents a broader category or type of product within H&M's inventory. Multiple articles may share the same product code if they belong to the same product line or style.

While both are unique identifiers, the article_id is specific to individual items, whereas the product_code represents a broader category or style of product.

Here is an example:

**Product: Basic T-Shirt**

- **Product Code:** TS001

- **Article IDs:**
    - Article ID: 1001 (Size: Small, Color: White)
    - Article ID: 1002 (Size: Medium, Color: White)
    - Article ID: 1003 (Size: Large, Color: White)
    - Article ID: 1004 (Size: Small, Color: Black)
    - Article ID: 1005 (Size: Medium, Color: Black)

In this example, "TS001" is the product code for the basic t-shirt style. Each variant of this t-shirt (e.g., different sizes and colors) has its own unique article_id.



In [6]:
articles_df = h_and_m_raw_data.extract_articles_df()
articles_df.shape

(105542, 25)

The articles DataFrame looks as follows:

In [7]:
articles_df.head(3)

article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
i64,i64,str,i64,str,str,i64,str,i64,str,i64,str,i64,str,i64,str,str,str,i64,str,i64,str,i64,str,str
108775015,108775,"""Strap top""",253,"""Vest top""","""Garment Upper body""",1010016,"""Solid""",9,"""Black""",4,"""Dark""",5,"""Black""",1676,"""Jersey Basic""","""A""","""Ladieswear""",1,"""Ladieswear""",16,"""Womens Everyday Basics""",1002,"""Jersey Basic""","""Jersey top with narrow shoulde…"
108775044,108775,"""Strap top""",253,"""Vest top""","""Garment Upper body""",1010016,"""Solid""",10,"""White""",3,"""Light""",9,"""White""",1676,"""Jersey Basic""","""A""","""Ladieswear""",1,"""Ladieswear""",16,"""Womens Everyday Basics""",1002,"""Jersey Basic""","""Jersey top with narrow shoulde…"
108775051,108775,"""Strap top (1)""",253,"""Vest top""","""Garment Upper body""",1010017,"""Stripe""",11,"""Off White""",1,"""Dusty Light""",9,"""White""",1676,"""Jersey Basic""","""A""","""Ladieswear""",1,"""Ladieswear""",16,"""Womens Everyday Basics""",1002,"""Jersey Basic""","""Jersey top with narrow shoulde…"


Check for NaNs:


In [8]:
articles_df.null_count()

article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,416


## Articles feature engineering


In [9]:
articles_df = compute_features_articles(articles_df)
articles_df.shape


(105542, 27)

The features of the articles look as:

In [10]:
articles_df.head(3)

article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,prod_name_length,article_description,image_url
str,i64,str,i64,str,str,i64,str,i64,str,i64,str,i64,str,i64,str,str,str,i64,str,i64,str,i64,str,u32,str,str
"""108775015""",108775,"""Strap top""",253,"""Vest top""","""Garment Upper body""",1010016,"""Solid""",9,"""Black""",4,"""Dark""",5,"""Black""",1676,"""Jersey Basic""","""A""","""Ladieswear""",1,"""Ladieswear""",16,"""Womens Everyday Basics""",1002,"""Jersey Basic""",9,"""Strap top - Vest top in Garmen…","""https://repo.hops.works/dev/jd…"
"""108775044""",108775,"""Strap top""",253,"""Vest top""","""Garment Upper body""",1010016,"""Solid""",10,"""White""",3,"""Light""",9,"""White""",1676,"""Jersey Basic""","""A""","""Ladieswear""",1,"""Ladieswear""",16,"""Womens Everyday Basics""",1002,"""Jersey Basic""",9,"""Strap top - Vest top in Garmen…","""https://repo.hops.works/dev/jd…"
"""108775051""",108775,"""Strap top (1)""",253,"""Vest top""","""Garment Upper body""",1010017,"""Stripe""",11,"""Off White""",1,"""Dusty Light""",9,"""White""",1676,"""Jersey Basic""","""A""","""Ladieswear""",1,"""Ladieswear""",16,"""Womens Everyday Basics""",1002,"""Jersey Basic""",13,"""Strap top (1) - Vest top in Ga…","""https://repo.hops.works/dev/jd…"


### Create embeddings from the articles description

In [11]:
for i, desc in enumerate(articles_df["article_description"].head(n=3)):
    logger.info(f"Item {i+1}:\n{desc}")

2026-06-08 22:43:13.529 | INFO     | __main__:<module>:2 - Item 1:
Strap top - Vest top in Garment Upper body
Appearance: Solid
Color: Dark Black (Black)
Category: Ladieswear - Womens Everyday Basics - Jersey Basic
Details: Jersey top with narrow shoulder straps.
2026-06-08 22:43:13.530 | INFO     | __main__:<module>:2 - Item 2:
Strap top - Vest top in Garment Upper body
Appearance: Solid
Color: Light White (White)
Category: Ladieswear - Womens Everyday Basics - Jersey Basic
Details: Jersey top with narrow shoulder straps.
2026-06-08 22:43:13.532 | INFO     | __main__:<module>:2 - Item 3:
Strap top (1) - Vest top in Garment Upper body
Appearance: Stripe
Color: Dusty Light White (Off White)
Category: Ladieswear - Womens Everyday Basics - Jersey Basic
Details: Jersey top with narrow shoulder straps.


In [12]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
logger.info(
    f"Loading '{settings.FEATURES_EMBEDDING_MODEL_ID}' embedding model to {device=}"
)

# Load the embedding model from SentenceTransformer's model registry.
model = SentenceTransformer(settings.FEATURES_EMBEDDING_MODEL_ID, device=device)

2026-06-08 22:43:14.619 | INFO     | __main__:<module>:8 - Loading 'all-MiniLM-L6-v2' embedding model to device='cpu'


2026-06-08 22:43:14,621 INFO: Load pretrained SentenceTransformer: all-MiniLM-L6-v2


In [13]:
articles_df = generate_embeddings_for_dataframe(
    articles_df, "article_description", model, batch_size=128
)  # Reduce batch size if getting OOM errors.

Generating embeddings: 100%|██████████| 105542/105542 [16:33:35<00:00,  1.77it/s]    


For each article description, we have a numerical vector which we can feed to a model, opposite to a string containing the description of an object.

In [14]:
articles_df[["article_description", "embeddings"]].head(3)

article_description,embeddings
str,list[f64]
"""Strap top - Vest top in Garmen…","[-0.026782, 0.082344, … 0.022782]"
"""Strap top - Vest top in Garmen…","[-0.010396, 0.089874, … 0.022564]"
"""Strap top (1) - Vest top in Ga…","[-0.032753, 0.091124, … 0.022804]"


### Looking at image links

In [15]:
articles_df["image_url"][0]

'https://repo.hops.works/dev/jdowling/h-and-m/images/010/0108775015.jpg'

In [16]:
from IPython.display import HTML, display

image_urls = articles_df["image_url"].tail(12).to_list()
grid_html = '<div style="display: grid; grid-template-columns: repeat(6, 1fr); gap: 10px; max-width: 900px;">'

for url in image_urls:
    grid_html += f'<img src="{url}" style="width: 100%; height: auto;">'

grid_html += "</div>"

display(HTML(grid_html))


## Customers Data

In [18]:
customers_df = h_and_m_raw_data.extract_customers_df()
customers_df.shape


(1371980, 7)

The customers DataFrame looks as follows:

In [19]:
customers_df.head(3)

customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
str,f64,f64,str,str,i64,str
"""00000dbacae5abe5e23885899a1fa4…",null,null,"""ACTIVE""","""NONE""",49,"""52043ee2162cf5aa7ee79974281641…"
"""0000423b00ade91418cceaf3b26c6a…",null,null,"""ACTIVE""","""NONE""",25,"""2973abc54daa8a5f8ccfe9362140c6…"
"""000058a12d5b43e67d225668fa1f8d…",null,null,"""ACTIVE""","""NONE""",24,"""64f17e6a330a85798e4998f62d0930…"


Check for NaNs:

In [20]:

customers_df.null_count()

customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
u32,u32,u32,u32,u32,u32,u32
0,895050,907576,6062,16009,15861,0


## Customers feature engineering


In [21]:
customers_df = compute_features_customers(customers_df, drop_null_age=True)
customers_df.shape

(1356119, 5)

The features of the customers DataFrame looks as follows:

In [22]:
customers_df.head(3)

customer_id,club_member_status,age,postal_code,age_group
str,str,f64,str,str
"""00000dbacae5abe5e23885899a1fa4…","""ACTIVE""",49.0,"""52043ee2162cf5aa7ee79974281641…","""46-55"""
"""0000423b00ade91418cceaf3b26c6a…","""ACTIVE""",25.0,"""2973abc54daa8a5f8ccfe9362140c6…","""19-25"""
"""000058a12d5b43e67d225668fa1f8d…","""ACTIVE""",24.0,"""64f17e6a330a85798e4998f62d0930…","""19-25"""



# Transactions Data

In [24]:
transactions_df = h_and_m_raw_data.extract_transactions_df()
transactions_df.shape

(31788324, 5)

The transaction DataFrame looks as follows:

In [25]:
transactions_df.head(3)

t_dat,customer_id,article_id,price,sales_channel_id
date,str,i64,f64,i64
2018-09-20,"""000058a12d5b43e67d225668fa1f8d…",663713001,0.050831,2
2018-09-20,"""000058a12d5b43e67d225668fa1f8d…",541518023,0.030492,2
2018-09-20,"""00007d2de826758b65a93dd24ce629…",505221004,0.015237,2


## Transactions feature engineering

In [26]:
transactions_df = compute_features_transactions(transactions_df)
transactions_df.shape

(31788324, 9)

The time of the year a purchase was made should be a strong predictor, as seasonality plays a big factor in fashion purchases. Here, you will use the month of the purchase as a feature. Since this is a cyclical feature (January is as close to December as it is to February), you'll map each month to the unit circle using sine and cosine.

Thus, the features of the transactions DataFrame look as follows:

In [27]:
transactions_df.head(3)

t_dat,customer_id,article_id,price,sales_channel_id,year,month,day,day_of_week
i64,str,str,f64,i64,i32,i8,i8,i8
0,"""000058a12d5b43e67d225668fa1f8d…","""663713001""",0.050831,2,2018,9,20,4
0,"""000058a12d5b43e67d225668fa1f8d…","""541518023""",0.030492,2,2018,9,20,4
0,"""00007d2de826758b65a93dd24ce629…","""505221004""",0.015237,2,2018,9,20,4


We don't want to work with ~30 million transactions in these series, as everything will take too much time to run. Thus, we create a subset of the original dataset by randomly sampling from the customers' datasets and taking only their transactions.

In [28]:
sampler = DatasetSampler(size=settings.CUSTOMER_DATA_SIZE)
dataset_subset = sampler.sample(
    customers_df=customers_df, transations_df=transactions_df
)
customers_df = dataset_subset["customers"]
transactions_df = dataset_subset["transactions"]

2026-06-09 18:55:19.952 | INFO     | recsys.features.customers:sample:29 - Sampling 1000 customers.
2026-06-09 18:55:20.932 | INFO     | recsys.features.customers:sample:32 - Number of transactions for all the customers: 31788324
2026-06-09 18:56:03.823 | INFO     | recsys.features.customers:sample:38 - Number of transactions for the 1000 sampled customers: 23799


In [29]:
transactions_df.shape

(23799, 9)

Some of the remaining customers:

In [30]:
for customer_id in transactions_df["customer_id"].unique().head(10):
    logger.info(f"Logging customer ID: {customer_id}")

2026-06-09 19:48:08.638 | INFO     | __main__:<module>:2 - Logging customer ID: 43b4a46e748b9fe4ac93e83d150c2ef77903c9fee6601ca61e29b00cc05ebb5f
2026-06-09 19:48:08.638 | INFO     | __main__:<module>:2 - Logging customer ID: 6d450227f6bff204f4237e88850e25e7cbeeeda2294a81bdd604a68c9d821050
2026-06-09 19:48:08.643 | INFO     | __main__:<module>:2 - Logging customer ID: 2703ad5b44313040b33e1306c0a370428ff19b5c1d95e0aa86d14ddec23bbd2b
2026-06-09 19:48:08.644 | INFO     | __main__:<module>:2 - Logging customer ID: d42df1d6e588463a06a3e3cdb8279ae32f1d610328bc9ecca9e20031c767d13e
2026-06-09 19:48:08.644 | INFO     | __main__:<module>:2 - Logging customer ID: 6d4c6e47fe5933f9d317263d19b5685223c4bedd160c1a6245a19112f0c28822
2026-06-09 19:48:08.650 | INFO     | __main__:<module>:2 - Logging customer ID: dd35d1ba370b9a6bb14ffcc0cdff99aa04b73cc0b1243e37920dc77c8f4b84ce
2026-06-09 19:48:08.652 | INFO     | __main__:<module>:2 - Logging customer ID: 7c329d8adfaa34a8b34c85b8941dd62e9cbf744a3c25bc2be4

# 🤳🏻 Interaction data

To train our models, we need more than just the transactions DataFrame. We need positive samples that signal whether a customer clicked or bought an item, but we also need negative samples that signal no interactions between a customer and an item.

In [31]:
interaction_df = generate_interaction_data(transactions_df)
interaction_df.shape

Processing customer chunks: 100%|██████████| 1/1 [00:09<00:00,  9.58s/it]


(134918, 5)

The interaction features look as follows:

In [32]:
interaction_df.head()

t_dat,customer_id,article_id,interaction_score,prev_article_id
i64,str,str,i64,str
-378000000,"""00b203a32faa3d007dba198ef27c15…","""590083014""",0,"""START"""
-370800000,"""00b203a32faa3d007dba198ef27c15…","""815434001""",0,"""590083014"""
-367200000,"""00b203a32faa3d007dba198ef27c15…","""292551014""",0,"""815434001"""
-363600000,"""00b203a32faa3d007dba198ef27c15…","""754925002""",0,"""292551014"""
-352800000,"""00b203a32faa3d007dba198ef27c15…","""590083014""",0,"""754925002"""


Let's take a look at the interaction score distribution:

In [33]:
interaction_df.group_by("interaction_score").agg(
    pl.count("interaction_score").alias("total_interactions")
)

interaction_score,total_interactions
i64,u32
0,73132
1,37987
2,23799


Here is what each score means:
- `0` : No interaction between a customer and an item
- `1` : A customer clicked an item
- `2` : A customer bought an item

# <span style="color:#ff5f27">🪄 Create Hopsworks Feature Groups </span>

A [feature group](https://docs.hopsworks.ai/feature-store-api/latest/generated/feature_group/) can be seen as a collection of conceptually related features.

To create a feature group you need to give it a name and specify a primary key. It is also best practice to provide a description of the contents of the feature group.

#### Customers

We set `online_enabled=True` to enable low-latency access to the data from the inference pipeline for real-time predictions. 

A full list of arguments can be found in the [documentation](https://docs.hopsworks.ai/feature-store-api/latest/generated/api/feature_store_api/#create_feature_group).

In [34]:
logger.info("Uploading 'customers' Feature Group to Hopsworks.")
customers_fg = feature_store.create_customers_feature_group(
    fs, df=customers_df, online_enabled=True
)

logger.info("✅ Uploaded 'customers' Feature Group to Hopsworks!")

2026-06-09 19:49:28.866 | INFO     | __main__:<module>:1 - Uploading 'customers' Feature Group to Hopsworks.


Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/fs/21731/fg/42416


Uploading Dataframe: 100.00% |██████████| Rows 1000/1000 | Elapsed Time: 00:02 | Remaining Time: 00:00


Launching job: customers_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/jobs/named/customers_1_offline_fg_materialization/executions
2026-06-09 19:49:54,950 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-06-09 19:49:58,235 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-06-09 19:52:03,055 INFO: Waiting for execution to finish. Current state: SUCCEEDING. Final status: UNDEFINED
2026-06-09 19:52:06,320 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-06-09 19:52:06,600 INFO: Waiting for log aggregation to finish.
2026-06-09 19:52:15,737 INFO: Execution finished successfully.


2026-06-09 19:52:22.035 | INFO     | __main__:<module>:6 - ✅ Uploaded 'customers' Feature Group to Hopsworks!


#### Articles

Let's do the same thing for the rest of the data frames.

In [35]:
logger.info("Uploading 'articles' Feature Group to Hopsworks.")
articles_fg = feature_store.create_articles_feature_group(
    fs,
    df=articles_df,
    articles_description_embedding_dim=model.get_sentence_embedding_dimension(),
    online_enabled=True,
)
logger.info("✅ Uploaded 'articles' Feature Group to Hopsworks!")


2026-06-09 20:59:53.426 | INFO     | __main__:<module>:1 - Uploading 'articles' Feature Group to Hopsworks.


Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/fs/21731/fg/42418


Uploading Dataframe: 100.00% |██████████| Rows 105542/105542 | Elapsed Time: 01:45 | Remaining Time: 00:00


Launching job: articles_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/jobs/named/articles_1_offline_fg_materialization/executions
2026-06-09 21:02:03,286 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-06-09 21:02:22,892 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-06-09 21:05:51,913 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-06-09 21:05:52,630 INFO: Waiting for log aggregation to finish.
2026-06-09 21:05:52,631 INFO: Execution finished successfully.


2026-06-09 21:05:52.633 | INFO     | __main__:<module>:8 - ✅ Uploaded 'articles' Feature Group to Hopsworks!


####  Transactions

In [36]:
logger.info("Uploading 'transactions' Feature Group to Hopsworks.")
trans_fg = feature_store.create_transactions_feature_group(
    fs=fs, df=transactions_df, online_enabled=True
)
logger.info("✅ Uploaded 'transactions' Feature Group to Hopsworks!")

2026-06-09 21:12:04.462 | INFO     | __main__:<module>:1 - Uploading 'transactions' Feature Group to Hopsworks.


Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/fs/21731/fg/42419


Uploading Dataframe: 100.00% |██████████| Rows 23799/23799 | Elapsed Time: 00:02 | Remaining Time: 00:00


Launching job: transactions_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/jobs/named/transactions_1_offline_fg_materialization/executions
2026-06-09 21:12:28,836 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-06-09 21:12:32,098 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-06-09 21:14:32,978 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-06-09 21:14:33,217 INFO: Waiting for log aggregation to finish.
2026-06-09 21:14:42,214 INFO: Execution finished successfully.


2026-06-09 21:14:55.437 | INFO     | __main__:<module>:5 - ✅ Uploaded 'transactions' Feature Group to Hopsworks!


#### Interactions

In [37]:
logger.info("Uploading 'interactions' Feature Group to Hopsworks.")
interactions_fg = feature_store.create_interactions_feature_group(
    fs=fs, df=interaction_df, online_enabled=True
)
logger.info("✅ Uploaded 'interactions' Feature Group to Hopsworks!!")

2026-06-09 21:56:26.362 | INFO     | __main__:<module>:1 - Uploading 'interactions' Feature Group to Hopsworks.


Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/fs/21731/fg/42420


Uploading Dataframe: 100.00% |██████████| Rows 134918/134918 | Elapsed Time: 00:03 | Remaining Time: 00:00


Launching job: interactions_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/jobs/named/interactions_1_offline_fg_materialization/executions
2026-06-09 21:56:50,870 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-06-09 21:56:54,136 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-06-09 21:59:14,620 INFO: Waiting for execution to finish. Current state: FINISHED. Final status: SUCCEEDED
2026-06-09 21:59:15,357 INFO: Waiting for log aggregation to finish.
2026-06-09 21:59:15,360 INFO: Execution finished successfully.


2026-06-09 21:59:21.685 | INFO     | __main__:<module>:5 - ✅ Uploaded 'interactions' Feature Group to Hopsworks!!


# Compute ranking dataset

The last step is to compute the ranking dataset used to train the scoring/ranking model from the feature groups we've just created:


In [38]:
ranking_df = compute_ranking_dataset(
    trans_fg,
    articles_fg,
    customers_fg,
)
ranking_df.shape

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (2.04s) 
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (9.63s) 
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.57s) 
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (163.72s) 


(224136, 15)

The ranking dataset looks as follows:

In [39]:
ranking_df.head(3)

customer_id,age,article_id,label,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name
str,f64,str,i32,str,str,str,str,str,str,str,str,str,str,str
"""34d30dcece38ac652e9fd05285d94d…",64.0,"""839496003""",1,"""Top""","""Garment Upper body""","""All over pattern""","""Beige""","""Dark""","""Beige""","""Jersey fancy""","""Ladieswear""","""Ladieswear""","""Womens Everyday Collection""","""Jersey Fancy"""
"""91a30617b9a5b5ff3927779204a176…",49.0,"""777093001""",1,"""Jumpsuit/Playsuit""","""Garment Full body""","""All over pattern""","""Dark Blue""","""Dark""","""Blue""","""Young Girl Dresses""","""Children Sizes 134-170""","""Baby/Children""","""Young Girl""","""Dresses/Skirts girls"""
"""23eeb5e9595c9409031f21a9c01fa3…",25.0,"""875719003""",1,"""Trousers""","""Garment Lower body""","""Solid""","""Beige""","""Dusty Light""","""Mole""","""Trousers DS""","""Divided""","""Divided""","""Divided Selected""","""Trousers"""


In [40]:
ranking_df.get_column("label").value_counts()

label,count
i32,u32
0,203760
1,20376


As the ranking dataset was computed based on articles, customers, and transactions Hopsworks Feature Groups, we can reflect this lineage in the ranking Feature Group.

In [41]:
logger.info("Uploading 'ranking' Feature Group to Hopsworks.")
rank_fg = feature_store.create_ranking_feature_group(
    fs,
    df=ranking_df,
    parents=[articles_fg, customers_fg, trans_fg],
    online_enabled=False
)
logger.info("✅ Uploaded 'ranking' Feature Group to Hopsworks!!")

2026-06-09 22:22:02.173 | INFO     | __main__:<module>:1 - Uploading 'ranking' Feature Group to Hopsworks.


Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/fs/21731/fg/42421


Uploading Dataframe: 100.00% |██████████| Rows 224136/224136 | Elapsed Time: 00:18 | Remaining Time: 00:00


Launching job: ranking_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/33047/jobs/named/ranking_1_offline_fg_materialization/executions
2026-06-09 22:22:41,134 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-06-09 22:22:44,396 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-06-09 22:25:01,788 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-06-09 22:25:02,049 INFO: Waiting for log aggregation to finish.
2026-06-09 22:25:18,129 INFO: Execution finished successfully.


2026-06-09 22:25:37.223 | INFO     | __main__:<module>:8 - ✅ Uploaded 'ranking' Feature Group to Hopsworks!!


## <span style="color:#ff5f27"> Inspecting the Feature Groups in Hopsworks UI </span>

View results in [Hopsworks Serverless](https://rebrand.ly/serverless-github): **Feature Store → Feature Groups**

---

In [42]:
notebook_end_time = time.time()
notebook_execution_time = notebook_end_time - notebook_start_time

logger.info(
    f"⌛️ Notebook Execution time: {notebook_execution_time:.2f} seconds ~ {notebook_execution_time / 60:.2f} minutes"
)

2026-06-10 20:27:15.750 | INFO     | __main__:<module>:4 - ⌛️ Notebook Execution time: 164858.92 seconds ~ 2747.65 minutes


# <span style="color:#ff5f27">→ Next Steps </span>

In the next notebook you'll train the retrieval model and register it to the Hopsworks model registry.